# Phishing Email Classifier — DistilBERT Fine-Tuning

This notebook trains a DistilBERT model to classify text as **phishing** or **legitimate**.

**Output**: HuggingFace-format model compatible with `PhishingClassifier` class in the NLP service.

**Target**: F1 >= 0.95 on validation set.

---

### Instructions
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. **(Optional)** Add your Hugging Face token in Colab Secrets as `HF_TOKEN` (key icon → Secrets) so gated datasets (e.g. TExtPhish) load and rate limits are higher.
3. Run all cells in order
4. Download the trained model from Google Drive or the output zip
5. Place files in `backend/ml-services/nlp-service/models/phishing-detector/`

In [ ]:
# Install dependencies
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from datasets import load_dataset

# Config
SEED = 42
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "/content/phishing-detector"
EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### Hugging Face login (for gated datasets and higher rate limits)

**In Colab:** Click the key icon in the left sidebar → **Secrets** → **Add new secret** → Name: `HF_TOKEN`, Value: *(paste your Hugging Face token)*. Then run the cell below.

In [ ]:
# Log in to Hugging Face (token from Colab Secrets as HF_TOKEN, or from env)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub. Gated datasets and higher rate limits enabled.")
else:
    print("No HF_TOKEN set. Add HF_TOKEN in Colab Secrets (key icon → Secrets) to use gated datasets (e.g. TExtPhish) and avoid rate limits.")

## 1. Load Dataset

We try multiple real phishing datasets from HuggingFace. You can also upload your own CSV with columns `text,label` (0=legitimate, 1=phishing).

In [ ]:
def load_phishing_data():
    """Load phishing email datasets from HuggingFace (currently supported: puyang2025, TExtPhish)."""
    all_samples = []

    # 1. puyang2025/phish-email-datasets — load each parquet file separately (avoids merge/read errors)
    puyang_files = ["Nazario.parquet", "Nigerian_Fraud.parquet", "Nazario_5.parquet", "Phishing_Email.parquet"]
    for fname in puyang_files:
        try:
            ds = load_dataset("puyang2025/phish-email-datasets", data_files=fname, split="train")
            before = len(all_samples)
            for row in ds:
                body = str(row.get("body", "") or "").strip()
                subject = str(row.get("subject", "") or "").strip()
                text = f"{subject}\n{body}".strip() or body or subject
                label = int(row.get("label", 0))
                if text and len(text) > 10:
                    all_samples.append((text[:2000], label))
            print(f"  puyang2025/{fname}: +{len(all_samples) - before} samples")
        except Exception as e:
            print(f"  puyang2025/{fname}: skipped ({e})")
    print(f"  Total from puyang2025: {len(all_samples)} samples")

    # 2. TExtPhish/TExtPhish (email-level) — gated; requires login + agreement on the Hub
    try:
        print("Loading TExtPhish/TExtPhish (email-level)...")
        ds = load_dataset("TExtPhish/TExtPhish", data_dir="email-level", split="train", sep=";")
        before = len(all_samples)
        for row in ds:
            text = str(row.get("content", "") or "").strip()
            label_str = str(row.get("label", "benign")).strip().lower()
            label = 0 if label_str == "benign" else 1
            if text and len(text) > 10:
                all_samples.append((text[:2000], label))
        print(f"  Added {len(all_samples) - before} samples from TExtPhish")
    except Exception as e:
        err = str(e)
        if "gated" in err.lower() or "authenticated" in err.lower():
            print("  TExtPhish is gated. To use: huggingface_hub.login() and accept terms at https://huggingface.co/datasets/TExtPhish/TExtPhish")
        else:
            print(f"  Could not load TExtPhish/TExtPhish: {e}")

    return all_samples


samples = load_phishing_data()
random.shuffle(samples)

texts = [s[0] for s in samples]
labels = [s[1] for s in samples]

print(f"\nTotal samples: {len(texts)}")
if texts:
    print(f"  Phishing:   {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
    print(f"  Legitimate: {len(labels) - sum(labels)} ({(len(labels)-sum(labels))/len(labels)*100:.1f}%)")
else:
    print("  No samples loaded. Use the optional CSV upload cell below or fix dataset access.")

### (Optional) Upload your own CSV dataset

If HuggingFace datasets are insufficient, upload a CSV with columns `text,label`.

In [ ]:
# Uncomment to upload your own CSV:
# from google.colab import files
# import csv
#
# uploaded = files.upload()
# for filename in uploaded:
#     with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
#         reader = csv.DictReader(f)
#         for row in reader:
#             text = row.get('text', row.get('content', row.get('body', '')))
#             label_raw = row.get('label', row.get('is_phishing', '0'))
#             label = 1 if str(label_raw).lower() in ('1', 'phishing', 'spam', 'true') else 0
#             if text and len(text.strip()) > 10:
#                 texts.append(text.strip()[:2000])
#                 labels.append(label)
#     print(f"After adding {filename}: {len(texts)} total samples")

## 2. Create Dataset & DataLoaders

In [ ]:
class PhishingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


if not texts:
    raise ValueError(
        "No samples loaded. Run the 'Load Dataset' cell above and ensure at least one HuggingFace dataset loads, "
        "or use the optional CSV upload cell to add your own data (columns: text, label)."
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = PhishingDataset(texts, labels, tokenizer, MAX_LENGTH)

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 3. Load Model & Configure Training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "legitimate", 1: "phishing"},
    label2id={"legitimate": 0, "phishing": 1},
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total training steps: {total_steps}")

## 4. Training Loop

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", pos_label=1
    )
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {
        "accuracy": accuracy, "precision": precision, "recall": recall,
        "f1": f1, "tpr": tpr, "fpr": fpr, "confusion_matrix": cm.tolist(),
    }


best_f1 = 0.0
best_metrics = {}

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == batch_labels).sum().item()
        total += batch_labels.size(0)

    val_metrics = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Train Acc: {correct/total:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val P: {val_metrics['precision']:.4f} | "
        f"Val R: {val_metrics['recall']:.4f}"
    )

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_metrics = val_metrics
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  -> Saved best model (F1: {best_f1:.4f})")

print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")

## 5. Save Metrics & Model Card

In [ ]:
# Save training metrics
metrics_serializable = {
    k: float(v) if isinstance(v, (float, np.floating)) else v
    for k, v in best_metrics.items()
}
with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
    json.dump(metrics_serializable, f, indent=2)

# Save model card
model_card = {
    "model_type": "phishing-classifier",
    "base_model": MODEL_NAME,
    "num_labels": 2,
    "labels": {"0": "legitimate", "1": "phishing"},
    "max_length": MAX_LENGTH,
    "metrics": metrics_serializable,
    "training_config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dataset_size": len(texts),
    },
}
with open(os.path.join(OUTPUT_DIR, "model_card.json"), "w") as f:
    json.dump(model_card, f, indent=2)

print("Saved training_metrics.json and model_card.json")
print(f"\nFinal metrics:")
for k, v in best_metrics.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 6. Verify Model Loads Correctly

In [ ]:
# Verify the saved model loads and runs inference
test_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
test_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
test_model.eval()

test_texts = [
    "Urgent: Your account will be suspended. Click here to verify immediately!",
    "Hi team, the meeting has been moved to 3 PM tomorrow. See you there.",
    "Your PayPal account has been limited. Verify at paypa1-secure.xyz now.",
    "Invoice #12345 is attached. Payment due by end of month.",
]

for text in test_texts:
    inputs = test_tokenizer(text, truncation=True, max_length=256, padding=True, return_tensors="pt")
    with torch.no_grad():
        logits = test_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
    phishing_prob = probs[0][1].item()
    prediction = "PHISHING" if phishing_prob > 0.5 else "LEGITIMATE"
    print(f"[{prediction}] (p={phishing_prob:.3f}) {text[:80]}...")

## 7. Download Model

Option A: Zip and download directly.  
Option B: Save to Google Drive.

In [ ]:
# Option A: Zip and download
!cd /content && zip -r phishing-detector.zip phishing-detector/

from google.colab import files
files.download("/content/phishing-detector.zip")

In [ ]:
# Option B: Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/phishing-detector /content/drive/MyDrive/phishing-detection-models/phishing-detector